# 10 — La libreria in un giro: PySharp
Stessa sequenza in **C#** e in **PySharp**, stessi dati (stesso seme) e stessi numeri: ogni riga `etichetta: valori` stampata qui compare identica nell'altro notebook, e il test `TourNotebooksAgreeAcrossLanguages` lo verifica. In C# si usa l'API **generica** (`T = double`); in PySharp la facciata `MathemaSharp.TimeSeries.Interop`, perché la sua reflection non passa `ReadOnlySpan<T>` né risolve i metodi generici.

Kernel **JupyterNet** `pysharp` (interprete Python in C#, `import numpy` è NDSharp) su runtime **.NET 11**. Parte II del libro: [la libreria](../../../refs/timeseries/01-libreria/README.md).

In [ ]:
import os, math, clr
import numpy as np

# La cartella di lavoro deve essere dentro il repository Numbers (scripts/run-notebooks.ps1 parte dalla radice).
root = os.getcwd()
while not os.path.exists(os.path.join(root, "src", "MathemaSharp.slnx")):
    parent = os.path.dirname(root)
    if parent == root:
        raise RuntimeError("Esegui il notebook con la cartella di lavoro dentro il repository MathemaSharp.")
    root = parent
clr.AddReference(os.path.join(root, "artifacts", "bin", "MathemaSharp.TimeSeries", "release", "MathemaSharp.TimeSeries.dll"))
from MathemaSharp.TimeSeries.Interop import Statistics, Forecasts, Metrics, Smoothing, Arima, Kalman, Var, Garch, Decomposition, Simulate

def P(label, values):
    if isinstance(values, (int, float)):
        values = [values]
    print(label + ": " + " ".join("%.6f" % v for v in values))

# Segnale simulato: AR(2) con phi = (0.6, -0.2), sigma = 1, 600 campioni. Ultimi 20 = test.
y = list(Simulate.Arma(2026, 600, 0.0, [0.6, -0.2], [], 1.0))
train, test = y[:580], y[580:]
P("n", len(y))

## 1 · Statistica di base (cap. 2)

In [ ]:
P("mean", Statistics.Mean(train))
P("acf", Statistics.Acf(train, 5))
P("pacf", Statistics.Pacf(train, 5))
lb = Statistics.LjungBox(train, 10, 0)
P("ljungbox", [lb.Statistic, lb.PValue])
P("quantili", [Statistics.Quantile(train, 0.1), Statistics.Quantile(train, 0.5), Statistics.Quantile(train, 0.9)])

## 2 · Riferimenti e metriche (cap. 3)

In [ ]:
for name, f in [("mean", Forecasts.Mean(train, 20)), ("naive", Forecasts.Naive(train, 20)), ("drift", Forecasts.Drift(train, 20))]:
    P("mase_" + name, Metrics.Mase(test, f, train, 1))

## 3 · AR e ARIMA (cap. 4)

In [ ]:
ar = Arima.FitYuleWalker(train, 2)
P("ar_phi", ar.Phi)
P("ar_sigma2", ar.InnovationVariance)
P("ar_forecast", ar.Forecast(train, 5))
arma = Arima.FitHannanRissanen(train, 1, 0, 1)
P("arma_params", [arma.Constant, arma.Phi[0], arma.Theta[0], arma.InnovationVariance])

## 4 · Smoothing esponenziale (cap. 5)

In [ ]:
alpha = Smoothing.FitSesAlpha(train)
P("ses_alpha", alpha)
P("ses_forecast", Smoothing.Ses(train, alpha, 3).Forecast)
P("holt_damped", Smoothing.Holt(train, 0.4, 0.2, 5, 0.9).Forecast)

# Serie con trend e stagionalita annuale (periodo 12), sullo stesso rumore.
seasonal = [v + 0.02 * t + 3 * math.sin(2 * math.pi * t / 12) for t, v in enumerate(y)]
hw = Smoothing.HoltWinters(seasonal[:588], 12, 0.3, 0.05, 0.2, 12, 1.0)
P("hw_forecast", list(hw.Forecast)[:6])
P("hw_mase", Metrics.Mase(seasonal[588:], list(hw.Forecast), seasonal[:588], 12))

## 5 · Filtro di Kalman (cap. 6)

In [ ]:
kf = Kalman.LocalLevel(0.1, 1.0, train[0], 10.0)
kr = Kalman.Filter(kf, train[:100])
P("kalman_gain_last", list(kr.FirstGain)[-1])
P("kalman_loglik", kr.LogLikelihood)
P("kalman_steady_gain", Kalman.SteadyStateGain(0.1, 1.0))

## 6 · VAR (cap. 7)
La matrice dei dati si passa **appiattita in ordine di riga**: `[x0, z0, x1, z1, ...]`.

In [ ]:
noise = [Simulate.Normal(7 + i, 1)[0] for i in range(400)]
flat = []
z_prev = 0.0
for t in range(400):
    z = 0.5 * noise[t] if t == 0 else 0.5 * z_prev + 0.3 * y[t - 1] + 0.5 * noise[t]
    flat.extend([y[t], z])
    z_prev = z
var1 = Var.Fit(flat, 400, 2, 1)
P("var_a1", Var.Coefficient(var1, 1))
P("var_stable", 1 if Var.IsStableOrder1Bivariate(var1) else 0)

## 7 · GARCH (cap. 8)

In [ ]:
truth = Garch.Model(0.05, 0.1, 0.85)
shocks = Simulate.Garch(7, truth, 1500)
fit = Garch.FitVarianceTargeting(shocks)
P("garch_fit", [fit.Omega, fit.Alpha, fit.Beta])
P("garch_forecast", fit.ForecastVariance(shocks, 5))

## 8 · STL (cap. 9)

In [ ]:
stl = Decomposition.Stl(seasonal, 12, 7, 2, 0)
P("stl_trend_ends", [stl.Trend[0], list(stl.Trend)[-1]])
P("stl_seasonal", list(stl.Seasonal)[:12])

## 9 · Solo in PySharp: la sovrapposizione con NumPy
Le stesse grandezze calcolate **solo con `numpy`** (NDSharp): la libreria e NumPy devono coincidere. Dove NumPy non basta (`np.correlate`, `np.linalg.eigvals`, `np.polyfit` non esistono in PySharp) si vede nel capitolo [Overlap con NumPy](../../../refs/timeseries/01-libreria/03-overlap-numpy.md).

In [ ]:
x = np.array(train)
n = len(x)
xc = x - np.mean(x)

# Autocovarianza (divisore n) con soli prodotti e somme di numpy.
gamma = np.array([np.sum(xc[:n - k] * xc[k:]) / n for k in range(6)])
lib_gamma = np.array(Statistics.Autocovariance(train, 5))
print("numpy_vs_lib_autocov_maxdiff: %.3e" % np.max(np.abs(gamma - lib_gamma)))

# Yule-Walker AR(2) risolto con np.linalg.solve invece di Levinson-Durbin.
G = np.array([[gamma[0], gamma[1]], [gamma[1], gamma[0]]])
phi_np = np.linalg.solve(G, gamma[1:3])
print("numpy_vs_lib_phi_maxdiff: %.3e" % np.max(np.abs(phi_np - np.array(ar.Phi))))

# Quantili: np.quantile (interpolazione lineare) coincide con Statistics.Quantile.
print("numpy_quantile_maxdiff: %.3e" % max([abs(float(np.quantile(x, p)) - Statistics.Quantile(train, p)) for p in [0.1, 0.5, 0.9]]))

# Differenza prima: np.diff coincide con Difference(lag=1); la differenza STAGIONALE si scrive a mano.
print("numpy_diff_vs_lib_maxdiff: %.3e" % np.max(np.abs(np.diff(x) - np.array(Statistics.Difference(train, 1)))))
print("numpy_seasonal_diff_vs_lib_maxdiff: %.3e" % np.max(np.abs((x[12:] - x[:-12]) - np.array(Statistics.Difference(train, 12)))))